# 00 — Colab Fresh Runtime Setup

Run this notebook first in a new Google Colab runtime. It clones the GitHub repository when needed, moves to the repository root, installs MuJoCo/OSMesa system packages, installs the Python package from the repo root, configures OSMesa rendering, and explicitly sets `PCDP_ARTIFACT_ROOT`.

Before running the setup cell, replace `GITHUB_REPO_URL` with this repository's GitHub clone URL if the default placeholder has not been updated.


In [ ]:
# @title Colab setup for phase_conditioned_diffusion_policy
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = 'phase_conditioned_diffusion_policy'
REPO_DIR = Path('/content') / REPO_NAME
GITHUB_REPO_URL = os.environ.get(
    'PCDP_REPO_URL',
    'https://github.com/YOUR_GITHUB_USERNAME/phase_conditioned_diffusion_policy.git',
)  # @param {type:'string'}

# Choose where experiment artifacts are written.
USE_GOOGLE_DRIVE = False  # @param {type:'boolean'}
DRIVE_ARTIFACT_ROOT = '/content/drive/MyDrive/phase_conditioned_diffusion_policy'  # @param {type:'string'}
LOCAL_ARTIFACT_ROOT = '/content/pcdp_artifacts'  # @param {type:'string'}

if not REPO_DIR.exists():
    if 'YOUR_GITHUB_USERNAME' in GITHUB_REPO_URL:
        raise ValueError(
            'Set GITHUB_REPO_URL to this repository\'s GitHub clone URL, '
            "or set os.environ[\'PCDP_REPO_URL\'] before running this cell."
        )
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'✓ repo already exists: {REPO_DIR}')

os.chdir(REPO_DIR)
try:
    get_ipython().run_line_magic('cd', str(REPO_DIR))
except NameError:
    pass

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run([
    'apt-get',
    'install',
    '-y',
    'libosmesa6-dev',
    'libgl1-mesa-glx',
    'libglfw3',
    'patchelf',
    '--quiet',
], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)

os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    artifact_root = Path(DRIVE_ARTIFACT_ROOT)
else:
    artifact_root = Path(LOCAL_ARTIFACT_ROOT)

artifact_root.mkdir(parents=True, exist_ok=True)
os.environ['PCDP_ARTIFACT_ROOT'] = str(artifact_root)

from pcdp.paths import ARTIFACT_ROOT, ensure_artifact_dirs
ensure_artifact_dirs()

print(f'✓ repo root: {REPO_DIR}')
print('✓ system packages installed for MuJoCo/OSMesa')
print('✓ pcdp installed from repo root (editable)')
print(f'✓ MUJOCO_GL={os.environ["MUJOCO_GL"]}')
print(f'✓ PYOPENGL_PLATFORM={os.environ["PYOPENGL_PLATFORM"]}')
print(f'✓ PCDP_ARTIFACT_ROOT={ARTIFACT_ROOT}')


Next notebooks to run: `01_data_preparation.ipynb → 02_vanilla_dp.ipynb → 03_phase_periodic.ipynb → 04_phase_trajectory.ipynb → 05_evaluation.ipynb`.
